In [6]:
!pip install playwright
!playwright install chromium

   ---------------------------------------- 0.0/38.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.2 MB 435.7 kB/s eta 0:01:28
   ---------------------------------------- 0.0/38.2 MB 326.8 kB/s eta 0:01:57
   ---------------------------------------- 0.1/38.2 MB 435.7 kB/s eta 0:01:28
   ---------------------------------------- 0.1/38.2 MB 535.8 kB/s eta 0:01:12
   ---------------------------------------- 0.1/38.2 MB 535.8 kB/s eta 0:01:12
   ---------------------------------------- 0.1/38.2 MB 535.8 kB/s eta 0:01:12
   ---------------------------------------- 0.1/38.2 MB 535.8 kB/s eta 0:01:12
   ---------------------------------------- 0.1/38.2 MB 327.4 kB/s eta 0:01:57
   ---------------------------------------- 0.1/38.2 MB 327.4 kB/s eta 0:01:57
   ---------------------------------------- 0.1/38.2 MB 327.4 kB/s eta 0:01:57



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\acer\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


|                                                                                |   0% of 191.8 MiB
|■■■■■■■■                                                                        |  10% of 191.8 MiB
|■■■■■■■■■■■■■■■■                                                                |  20% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■                                                        |  30% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                                |  40% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                        |  50% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                |  60% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                        |  70% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                |  80% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■        |  90% of 

In [13]:
!pip install nest_asyncio


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\acer\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
import asyncio
import pandas as pd
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor
from playwright.async_api import async_playwright

# 1. ตั้งค่า Timezone
BKK_TZ = ZoneInfo("Asia/Bangkok")

def get_bkk_now():
    return datetime.now(BKK_TZ)

def get_thai_date_str(dt):
    thai_months = ["มกราคม", "กุมภาพันธ์", "มีนาคม", "เมษายน", "พฤษภาคม", "มิถุนายน",
                   "กรกฎาคม", "สิงหาคม", "กันยายน", "ตุลาคม", "พฤศจิกายน", "ธันวาคม"]
    thai_year = dt.year + 543
    return f"{dt.day:02d} {thai_months[dt.month - 1]} {thai_year}"

def parse_thai_dt(raw_date, raw_time):
    try:
        d, m_name, y = raw_date.split()
        thai_months = ["มกราคม", "กุมภาพันธ์", "มีนาคม", "เมษายน", "พฤษภาคม", "มิถุนายน",
                       "กรกฎาคม", "สิงหาคม", "กันยายน", "ตุลาคม", "พฤศจิกายน", "ธันวาคม"]
        m = thai_months.index(m_name) + 1
        y_iso = int(y) - 543
        t_str = raw_time.replace("น.", "").strip()
        if not t_str or t_str == "ไม่ระบุ": t_str = "00:00"
        return datetime(y_iso, m, int(d), int(t_str.split(':')[0]), int(t_str.split(':')[1]))
    except: return None

print("✅ Cell 1: โหลด Library และตั้งค่าเวลาสำเร็จ!")

✅ Cell 1: โหลด Library และตั้งค่าเวลาสำเร็จ!


In [14]:
async def _internal_scrape_js100(start_window, end_window):
    start_window_naive = start_window.replace(tzinfo=None)
    end_window_naive = end_window.replace(tzinfo=None)
    
    keywords = ["เมืองปทุมธานี", "คลองหลวง", "ธัญบุรี", "ลำลูกกา", "สามโคก", "ลาดหลุมแก้ว", "หนองเสือ", "รังสิต"
]
    all_data = []
    now = get_bkk_now()
    today_str = get_thai_date_str(now)
    yesterday_str = get_thai_date_str(now - timedelta(days=1))

    print(f"🚀 เริ่มดึงข้อมูลช่วง: {start_window_naive.strftime('%Y-%m-%d %H:%M')} ถึง {end_window_naive.strftime('%Y-%m-%d %H:%M')}")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--disable-blink-features=AutomationControlled"])
        context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
        page = await context.new_page()

        for keyword in keywords:
            print(f"🔍 กำลังค้นหาคำว่า: {keyword}...")
            try:
                await page.goto("https://www.js100.com/en/site/home/search_advance", wait_until="domcontentloaded", timeout=20000)
                
                search_input = await page.wait_for_selector('input[name="search_text"], #search_result_input', timeout=10000)
                await search_input.fill(keyword)
                await search_input.press("Enter")
                
                await page.wait_for_selector('#search_result_list li', timeout=15000)
                await asyncio.sleep(1)
                
                items = await page.query_selector_all('#search_result_list li')
                valid_count = 0
                
                for item in items:
                    h4_tag = await item.query_selector('h4')
                    raw_datetime = (await h4_tag.inner_text()).strip() if h4_tag else ""
                    
                    date_part = raw_datetime.split(",")[0].strip() if "," in raw_datetime else raw_datetime
                    time_part = raw_datetime.split(",")[1].strip() if "," in raw_datetime else "00:00"
                    
                    if "วันนี้" in date_part: date_part = today_str
                    elif "เมื่อวาน" in date_part: date_part = yesterday_str
                    
                    dt = parse_thai_dt(date_part, time_part)
                    
                    if dt and (start_window_naive <= dt <= end_window_naive):
                        a_tag = await item.query_selector('a')
                        p_tag = await item.query_selector('p')
                        
                        if a_tag and (await a_tag.inner_text()).strip():
                            headline = (await a_tag.inner_text()).strip()
                            link = await a_tag.get_attribute('href')
                            full_link = f"https://www.js100.com{link}" if link and link.startswith('/') else link
                            category = "ข่าว"
                            content = f"{headline} (รายละเอียด: {full_link})"
                        elif p_tag:
                            category = "ข้อมูลจราจร"
                            content = (await p_tag.inner_text()).strip()
                        else:
                            category = "อื่นๆ"
                            content = (await item.inner_text()).replace(raw_datetime, "").strip()

                        all_data.append({
                            "search_keyword": keyword, 
                            "timestamp": dt, 
                            "category": category, 
                            "content": content
                        })
                        valid_count += 1
                        
                print(f"✅ [{keyword}] ดึงข้อมูลสำเร็จ: {valid_count} รายการ")

            except Exception as e:
                print(f"⚠️ ดึงข้อมูล '{keyword}' ไม่สำเร็จ Error: {e}")
                continue

        await browser.close()
        
    return pd.DataFrame(all_data)

print("✅ Cell 2: โหลดฟังก์ชัน Scraper สำเร็จ!")

✅ Cell 2: โหลดฟังก์ชัน Scraper สำเร็จ!


In [12]:
def _run_in_new_loop(start_window, end_window):
    # บังคับใช้ ProactorEventLoop บน Thread แยกสำหรับ Windows
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    new_loop = asyncio.new_event_loop()
    asyncio.set_event_loop(new_loop)
    try:
        return new_loop.run_until_complete(_internal_scrape_js100(start_window, end_window))
    finally:
        new_loop.close()

async def test_scrape_js100(start_window, end_window):
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=1) as pool:
        return await loop.run_in_executor(pool, _run_in_new_loop, start_window, end_window)

print("✅ Cell 3: ตั้งค่า Thread Wrapper สำเร็จ!")

✅ Cell 3: ตั้งค่า Thread Wrapper สำเร็จ!


In [15]:
# 1. กำหนดช่วงเวลาเป้าหมาย (เมื่อวานนี้ 00:00 - 23:59)
now = get_bkk_now()
yesterday = now - timedelta(days=1)

start_time = yesterday.replace(hour=0, minute=0, second=0, microsecond=0)
end_time = yesterday.replace(hour=23, minute=59, second=59, microsecond=999999)

print(f"🎯 กำหนดเป้าหมายดึงข้อมูล: {start_time.strftime('%d/%m/%Y %H:%M:%S')} ถึง {end_time.strftime('%d/%m/%Y %H:%M:%S')}")

# 2. เรียกใช้งานฟังก์ชัน
df_lab = await test_scrape_js100(start_time, end_time)

# 3. คลีนข้อมูลและจัดเรียง
if not df_lab.empty:
    df_lab = df_lab.drop_duplicates(subset=['timestamp', 'content'], keep='first')
    df_lab = df_lab.sort_values(by='timestamp').reset_index(drop=True)

# 4. แสดงผลลัพธ์
print(f"\n📊 สรุป: ดึงข้อมูลได้ทั้งหมด {len(df_lab) if not df_lab.empty else 0} รายการ")
if not df_lab.empty:
    display(df_lab.head(10))

🎯 กำหนดเป้าหมายดึงข้อมูล: 19/08/2026 00:00:00 ถึง 19/08/2026 23:59:59
🚀 เริ่มดึงข้อมูลช่วง: 2026-08-19 00:00 ถึง 2026-08-19 23:59
🔍 กำลังค้นหาคำว่า: เมืองปทุมธานี...
✅ [เมืองปทุมธานี] ดึงข้อมูลสำเร็จ: 0 รายการ
🔍 กำลังค้นหาคำว่า: คลองหลวง...
✅ [คลองหลวง] ดึงข้อมูลสำเร็จ: 1 รายการ
🔍 กำลังค้นหาคำว่า: ธัญบุรี...
✅ [ธัญบุรี] ดึงข้อมูลสำเร็จ: 0 รายการ
🔍 กำลังค้นหาคำว่า: ลำลูกกา...
✅ [ลำลูกกา] ดึงข้อมูลสำเร็จ: 1 รายการ
🔍 กำลังค้นหาคำว่า: สามโคก...
✅ [สามโคก] ดึงข้อมูลสำเร็จ: 0 รายการ
🔍 กำลังค้นหาคำว่า: ลาดหลุมแก้ว...
✅ [ลาดหลุมแก้ว] ดึงข้อมูลสำเร็จ: 0 รายการ
🔍 กำลังค้นหาคำว่า: หนองเสือ...
✅ [หนองเสือ] ดึงข้อมูลสำเร็จ: 0 รายการ
🔍 กำลังค้นหาคำว่า: รังสิต...
✅ [รังสิต] ดึงข้อมูลสำเร็จ: 1 รายการ

📊 สรุป: ดึงข้อมูลได้ทั้งหมด 3 รายการ


,search_keyword,timestamp,category,content
0,ลำลูกกา,2026-08-19 07:00:00,ข้อมูลจราจร,อุบัติเหตุ รถSUVกับรถเก๋งชนกัน ถนนลำลูกกา ขาเข...
1,คลองหลวง,2026-08-19 11:08:00,ข้อมูลจราจร,เหตุเพลิงไหม้รถยนต์ มอเตอร์เวย์ สาย 9 จากแยกไป...
2,รังสิต,2026-08-19 17:13:00,ข้อมูลจราจร,ถนนวิภาวดีรังสิต ขาออก ช่วงแยกลาดพร้าว มุ่งหน้...
